In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
import geopandas as gpd


from time import perf_counter as perf
import pyodbc
import urllib
import sqlalchemy as sqla

# pd.options.display.float_format = '{:.0f}'.format
pd.set_option('display.max_columns', None)

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'RTIS Data')
path_main = os.path.join(path_sp, 'Data')
path_congestion = os.path.join(path_main, 'Safe Equitable Resilient Infrastructure', 'Congestion')

# Git
if user == 'jfontes':
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

if user in ['jchoy', 'AAlAzzawi']:
    path_git = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')
path_config0 = os.path.join(path_git, 'config')
path_code    = os.path.join(path_git, 'Python Code', 'RTIS')
path_config  = os.path.join(path_code, 'config')
path_sql     = os.path.join(path_git, 'Python Code', 'RTIS', 'SQL Scripts')
path_geo  = os.path.join(path_users, 'Documents', 'Geospatial Data')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

In [ ]:
# Keep track of time amounted while SQL query is running through all years
start_time = time.time()

# Set up parameters for SQL queries
db = 'NPMRDS'

print('Creating query string from .sql file...')

# tmc_code = '105-16661'
# tmc_code = '105P17076'
# tmc_code = '105-16662'
tmc_code = '105+05244'
# tmc_code = '105P17073'

query_string = f"SELECT * FROM npmrds_2019_alltmc_paxtruck_comb WHERE tmc_code = '{tmc_code}'"
df_sql19 = sqlqry_to_df(query_string, db)

query_string = f"SELECT * FROM npmrds_2023_alltmc_paxtruck_comb WHERE tmc_code = '{tmc_code}'"
df_sql23 = sqlqry_to_df(query_string, db)


print(f"Process complete.  It took --- {round((time.time() - start_time)/60, 1)} minutes ---")
print('')


display(df_sql19.head(), df_sql23.head())

In [ ]:
# 2023 and 2022 tables might be missing a lot of data
# There doesn't seem to be consistent 15 minute interval data going on here

print(df_sql19.shape[0])
print(df_sql23.shape[0])

print(df_sql19.measurement_tstamp.duplicated().sum())
print(df_sql23.measurement_tstamp.duplicated().sum())

print(df_sql19.travel_time_seconds.sum())
print(df_sql23.travel_time_seconds.sum())

df_sql19 = df_sql19.sort_values(['measurement_tstamp']).reset_index(drop = True)
df_sql23 = df_sql23.sort_values(['measurement_tstamp']).reset_index(drop = True)
display(df_sql19.head(15), df_sql23.head(15))

In [ ]:
# Keep track of time amounted while SQL query is running through all years
start_time = time.time()

# Set up parameters for SQL queries
db = 'NPMRDS'

print('Creating query string from .sql file...')

query_string = "SELECT * FROM npmrds_2019_all_tmcs_txt"
df_sql19_txt = sqlqry_to_df(query_string, db)

query_string = "SELECT * FROM npmrds_2023_alltmc_txt"
df_sql23_txt = sqlqry_to_df(query_string, db)

print(f"Process complete.  It took --- {round((time.time() - start_time)/60, 1)} minutes ---")
print('')

display(df_sql19_txt.head(), df_sql23_txt.head())

In [ ]:
print(df_sql19_txt.miles.sum())
print(df_sql23_txt.miles.sum())

In [ ]:
gpd_ca19 = gpd.read_file(os.path.join(path_geo, 'RTIS', 'California 2019', 'California.shp'))
gpd_ca23 = gpd.read_file(os.path.join(path_geo, 'RTIS', 'California 2023', 'California.shp'))

display(gpd_ca19.head(3), gpd_ca23.head(3))

In [ ]:
gdf_counties = gpd.read_file(os.path.join(path_geo, 'TIGER', 'tl_2022_us_county_SACOG.geojson'))
gdf_counties = gdf_counties[['COUNTYFP', 'NAME', 'geometry']]
gdf_counties = gdf_counties.to_crs(crs='4326')
gdf_counties.head()

In [ ]:
gdf_int19 = gpd.overlay(gpd_ca19, gdf_counties, how='intersection')
gdf_int23 = gpd.overlay(gpd_ca23, gdf_counties, how='intersection')

gpd_ca19 = gpd_ca19[gpd_ca19['Tmc'].isin(gdf_int19['Tmc'].values)]
gpd_ca23 = gpd_ca23[gpd_ca23['Tmc'].isin(gdf_int23['Tmc'].values)]

gpd_ca19 = gpd_ca19.reset_index(drop = True)
gpd_ca23 = gpd_ca23.reset_index(drop = True)

# gpd_ca19.to_file(os.path.join(path_geo, 'RTIS', 'SACOG roads RTIS 2019.geojson'), driver='GeoJSON')
# gpd_ca23.to_file(os.path.join(path_geo, 'RTIS', 'SACOG roads RTIS 2023.geojson'), driver='GeoJSON')

In [ ]:
print(gpd_ca19.Miles.sum())
print(gpd_ca23.Miles.sum())

In [ ]:
display(gpd_ca19.head(3), gpd_ca23.head(3))

In [ ]:
# # Average free flow speed by Year and road type
# df_ff1 = df_congestion2.groupby(['Year', 'f_system'], as_index = False)['ff_speed_art60thp'].mean()
# df_ff1 = df_ff1.sort_values(['Year', 'f_system'])
# display(df_ff1)
# # 2022/2023 doesn't have f_system 1?


# # Average free flow speed by year
# display(df_congestion2.groupby(['Year'], as_index = False)['ff_speed_art60thp'].mean())
# # free flow speeds in 2023 are lower than pre-covid levels?